# 04 — Sensors, Retries, Dynamic DAGs & MWAA Patterns

Covers sensors, SLA vs execution_timeout, dynamic task generation, and what's different on MWAA.

## 1. Sensors — mode matters more than anything

In [ ]:
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.sensors.filesystem import FileSensor  # swap for S3KeySensor on MWAA

# mode="poke"       → holds a worker slot the ENTIRE wait period (BAD for long waits)
# mode="reschedule" → releases worker slot between checks (ALWAYS use for prod)

with DAG("sensor_demo", start_date=datetime(2024,1,1), catchup=False) as dag:

    wait = FileSensor(
        task_id="wait_for_file",
        filepath="/tmp/{{ ds_nodash }}/data.parquet",
        mode="reschedule",   # releases worker slot between pokes
        poke_interval=120,   # check every 2 minutes
        timeout=4 * 3600,    # fail after 4 hours
        soft_fail=True,      # SKIPPED on timeout instead of FAILED
    )

    process = PythonOperator(
        task_id="process",
        python_callable=lambda ds, **kw: print(f"Processing {ds}"),
        trigger_rule="none_failed_min_one_success",
    )

    wait >> process

print("Sensor DAG built")

**Poke vs Reschedule — interview answer:**
> *"mode=poke holds a worker slot for the entire wait — with a 4-hour timeout and 10 sensors running, you'd exhaust the worker pool. mode=reschedule returns the slot between checks; the scheduler re-queues the sensor after each interval. I always use reschedule in prod. Used this in A2H to wait for upstream Glue job manifests before triggering the classification batch."*

## 2. ExternalTaskSensor — waiting on another DAG

In [ ]:
# from airflow.sensors.external_task import ExternalTaskSensor

# ExternalTaskSensor(
#     task_id="wait_for_ingestion",
#     external_dag_id="ingestion_pipeline",
#     external_task_id="final_task",        # None = wait for whole DAG
#     execution_delta=timedelta(hours=1),   # upstream runs 1hr before this DAG
#     mode="reschedule",
#     timeout=3600,
# )

print("ExternalTaskSensor pattern above (commented)")

## 3. Retries & Error Handling

In [ ]:
def alert_on_failure(context):
    dag_id   = context["dag"].dag_id
    task_id  = context["task"].task_id
    exec_date = context["ds"]
    exception = context.get("exception")
    print(f"FAILED: {dag_id}.{task_id} on {exec_date} | error: {exception}")
    # In real MWAA: post to SNS or Slack

default_args_with_retry = {
    "owner": "kanishk",
    "retries": 3,
    "retry_delay": timedelta(minutes=2),
    "retry_exponential_backoff": True,   # 2m → 4m → 8m
    "max_retry_delay": timedelta(minutes=60),
    "execution_timeout": timedelta(hours=2),   # KILLS the task if it hangs
    "on_failure_callback": alert_on_failure,
}

with DAG("retry_demo", default_args=default_args_with_retry,
         start_date=datetime(2024,1,1), catchup=False) as dag:

    critical = PythonOperator(
        task_id="critical_task",
        python_callable=lambda: print("running"),
        sla=timedelta(hours=1),          # alert if not done in 1hr — doesn't kill the task
        execution_timeout=timedelta(hours=2),  # actually kills it at 2hr
    )

print("Retry DAG built")

**SLA vs execution_timeout — interview answer:**
> *"SLA fires an alert if the task isn't done within a time window from DAG start, but doesn't stop it — it's a business-level alert. execution_timeout actually kills the task if it exceeds the limit. In A2H I set execution_timeout on the Glue operator to prevent runaway jobs from blocking the queue, and SLA to alert ops if the pipeline wasn't done before the 9am business reporting window."*

## 4. Dynamic DAGs

In [ ]:
from airflow.utils.task_group import TaskGroup

REGIONS = ["na", "eu", "fe"]

def process_region(region, ds=None, **kwargs):
    print(f"Processing {region} for {ds}")

with DAG("dynamic_regional", start_date=datetime(2024,1,1), catchup=False) as dag:

    with TaskGroup("regional_processing") as regional_group:
        for region in REGIONS:
            PythonOperator(
                task_id=f"process_{region}",
                python_callable=process_region,
                op_kwargs={"region": region},
            )

    aggregate = PythonOperator(
        task_id="aggregate",
        python_callable=lambda: print("aggregating all regions"),
    )

    regional_group >> aggregate   # all 3 regional tasks run in parallel, then aggregate

print(f"Tasks: {[t.task_id for t in dag.tasks]}")

In [ ]:
# Airflow 2.3+ — dynamic task mapping (cleaner for variable-length inputs)
from airflow.decorators import dag, task

@dag(schedule="@daily", start_date=datetime(2024, 1, 1), catchup=False)
def dynamic_mapping():

    @task
    def get_regions() -> list[str]:
        return ["na", "eu", "fe", "jp"]   # could come from a DB or S3

    @task
    def process(region: str):
        print(f"Processing {region}")

    regions = get_regions()
    process.expand(region=regions)   # one task per region, created at runtime

dm = dynamic_mapping()
print("Dynamic mapping DAG:", dm.dag_id)

## 5. MWAA vs Local Airflow

| | Local | MWAA |
|---|---|---|
| DAG deployment | Copy to `dags/` | Upload to S3, syncs every ~30s |
| Dependencies | `pip install` | `requirements.txt` in S3 |
| Secrets | `airflow.cfg` / env vars | AWS Secrets Manager |
| Connections | UI or env vars | `/airflow/connections/<name>` in Secrets Manager |
| Variables | UI | `/airflow/variables/<name>` in Secrets Manager |
| Logs | Local files | CloudWatch Logs |
| Scaling | Manual | Managed min/max workers |

**Secrets Manager convention:**
```
/airflow/connections/aws_default     → Airflow connection
/airflow/connections/redshift_conn   → Redshift connection
/airflow/variables/environment       → "prod"
/airflow/variables/pipeline_config   → JSON blob
```

**Your A2H MWAA pattern (interview):**
> *"In A2H we used MWAA with max_active_runs=1 because the pipeline is stateful — overlapping runs write to the same S3 prefix and corrupt the delta. We migrated from Step Functions because MWAA's Clear button let us rerun from any failed task without touching upstream, cutting recovery time from 45 minutes to 8 minutes."*